# $(SASA) Models - Kmeans$

In [1]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import pickle
import numpy as np
import pandas as pd

from pmbrl.model2 import Base_Line_Simple_Model
# from pmbrl.model2 import Model, Regularized_Reference_Loss
from pmbrl.data import Experiment_Data, get_data_expanded, get_data_compacted

In [2]:
nome_do_arquivo = 'kmodels.pkl'

with open(nome_do_arquivo, 'rb') as arquivo:
    exp = pickle.load(arquivo)
    data = exp['data']
    models = exp['model']

del exp
del arquivo

In [3]:
data = Experiment_Data()

data.load(path='../testing_data.csv')

expansions = {
    's': ['s0', 's1', 's2', 's3'],
    's_': ['s_0', 's_1', 's_2', 's_3'],
    's__': ['s__0', 's__1', 's__2', 's__3'],
}

df = get_data_expanded(data.build_training_dataset(), expansions)
# df = df.loc[df['episode']<15].copy()
df.head()

,step,episode,p,s,a,r,s_,a_,r_,s__,...,s2,s3,s_0,s_1,s_2,s_3,s__0,s__1,s__2,s__3
469,35,26,"(0.1168448454824361, 0.6199561479617748)","(-0.189, 0.546, 0.112, -0.574)",1,1.0,"(-0.179, 0.737, 0.101, -0.783)",1.0,1.0,"(-0.164, 0.929, 0.085, -0.995)",...,0.112,-0.574,-0.179,0.737,0.101,-0.783,-0.164,0.929,0.085,-0.995
1835,1,91,"(0.1501612193777863, 0.2709280492756518)","(-0.021, -0.167, 0.032, 0.559)",1,1.0,"(-0.024, 0.039, 0.043, 0.018)",0.0,1.0,"(-0.023, -0.169, 0.043, 0.603)",...,0.032,0.559,-0.024,0.039,0.043,0.018,-0.023,-0.169,0.043,0.603
213,2,14,"(0.4344834695485677, 0.3283152435126403)","(0.028, -0.432, 0.034, 1.242)",1,1.0,"(0.02, -0.225, 0.059, 0.674)",0.0,1.0,"(0.015, -0.435, 0.072, 1.299)",...,0.034,1.242,0.020,-0.225,0.059,0.674,0.015,-0.435,0.072,1.299
1856,8,92,"(0.2225177737903001, 0.4165106988169785)","(0.035, 0.36, -0.083, -0.883)",1,1.0,"(0.043, 0.561, -0.101, -1.301)",0.0,1.0,"(0.054, 0.364, -0.127, -0.959)",...,-0.083,-0.883,0.043,0.561,-0.101,-1.301,0.054,0.364,-0.127,-0.959
324,20,20,"(0.5511096795536096, 0.8938255112494718)","(-0.114, -0.383, 0.043, 0.352)",1,1.0,"(-0.122, -0.191, 0.05, 0.119)",1.0,1.0,"(-0.126, 0.001, 0.053, -0.111)",...,0.043,0.352,-0.122,-0.191,0.050,0.119,-0.126,0.001,0.053,-0.111


# Predict 

In [4]:
def optim_params(prediction_dataset):
    weights = prediction_dataset.copy()
    for m, _ in enumerate(models):
        weights['estimated_s'] = prediction_dataset[f'estimated_s_model_{m}']
        expansions = {f'estimated_s': [f'estimated_s0', f'estimated_s1', f'estimated_s2', f'estimated_s3']}
        ds = get_data_expanded(weights, expansions)
        # weights[f'estimated_s_model_{m}'] = ds['estimated_s']
        for d in range(4):
            weights[f'estimated_s{d}_model{m}'] = ds[f'estimated_s{d}']
    weights['A'] = weights.apply(lambda row: np.array([[row[f'estimated_s{d}_model{m}'] for m,_ in enumerate(models)] for d in range(4)]),axis=1)
    weights['b'] = weights.apply(lambda row: np.array([row[f's__{d}'] for d in range(4)]),axis=1)
    weights['w'] = weights.apply(lambda row: np.linalg.lstsq(row.A, row.b)[0],axis=1)

    weights[[f'estimated_weight_{m}' for m, _ in enumerate(models)]] = weights.apply(lambda row: pd.Series(row['w']),axis=1)
    return weights


In [5]:

def predict_with_params(prediction_dataset, weights):
    for m, _ in enumerate(models):
        prediction_dataset[f'weight_{m}'] = weights[f'estimated_weight_{m}']

    expansions = {
        f'estimated_s_model_{m}': [f'estimated_s{d}_model{m}' for d in range(4)] for m,_ in enumerate(models)}
    ds = get_data_expanded(prediction_dataset, expansions)
    
    for d in range(4):
        ds[f'estimated_weighted_s{d}'] = ds.apply(
            lambda row: np.sum([row[f'estimated_s{d}_model{m}']* row[f'weight_{m}'] for m, _ in enumerate(models)])
            , axis=1
        )
    
    ds = get_data_compacted(ds, {'estimated_weighted_s': [f'estimated_weighted_s{d}' for d in range(4)]})
     
    prediction_dataset['estimated_r'] = ds['estimated_r_model_0']
    prediction_dataset['estimated_s'] = ds['estimated_weighted_s']
    
    results = data.get_evaluation_metrics(prediction_dataset, p=False)
    prediction_dataset[f'rse'] = results['rse']
    prediction_dataset[f'rse_normalized'] = results['rse_normalized']

    prediction_dataset[f'rse_s0'] = results['rse_s0']
    prediction_dataset[f'rse_s1'] = results['rse_s1']
    prediction_dataset[f'rse_s2'] = results['rse_s2']
    prediction_dataset[f'rse_s3'] = results['rse_s3']

    prediction_dataset[f'rse_s0_normalized'] = results['rse_s0_normalized']
    prediction_dataset[f'rse_s1_normalized'] = results['rse_s1_normalized']
    prediction_dataset[f'rse_s2_normalized'] = results['rse_s2_normalized']
    prediction_dataset[f'rse_s3_normalized'] = results['rse_s3_normalized']

    return prediction_dataset

In [6]:
def evaluate(models, df):
    def predict(model):
        prediction_dataset = df.copy()
        prediction_dataset[model.grouped_targets_lables] = prediction_dataset.apply(lambda row: data._predict_from_row(row, model), axis=1, result_type='expand')
        return prediction_dataset

    predictions = [predict(m) for m in models]

    prediction_dataset = df.copy()
    for i, pred in enumerate(predictions):
        prediction_dataset[f'estimated_r_model_{i}'] = pred['estimated_r']
        prediction_dataset[f'estimated_s_model_{i}'] = pred['estimated_s']
        
        results = data.get_evaluation_metrics(pred, p=False)
        prediction_dataset[f'rse_model_{i}'] = results['rse']
        prediction_dataset[f'rse_normalized_model_{i}'] = results['rse_normalized']

        prediction_dataset[f'rse_s0_model_{i}'] = results['rse_s0']
        prediction_dataset[f'rse_s1_model_{i}'] = results['rse_s1']
        prediction_dataset[f'rse_s2_model_{i}'] = results['rse_s2']
        prediction_dataset[f'rse_s3_model_{i}'] = results['rse_s3']

        prediction_dataset[f'rse_s0_normalized_model_{i}'] = results['rse_s0_normalized']
        prediction_dataset[f'rse_s1_normalized_model_{i}'] = results['rse_s1_normalized']
        prediction_dataset[f'rse_s2_normalized_model_{i}'] = results['rse_s2_normalized']
        prediction_dataset[f'rse_s3_normalized_model_{i}'] = results['rse_s3_normalized']


    return prediction_dataset

In [7]:
def predicts(models, df):
    pre_df = df.copy()
    pre_df[['s_', 's_0', 's_1', 's_2', 's_3', 'a_', 's__0', 's__1', 's__2', 's__3', 's__']] = pre_df[['s', 's0', 's1', 's2', 's3', 'a', 's_0', 's_1', 's_2', 's_3', 's_']]

    prediction_dataset = evaluate(models, pre_df)
    params = optim_params(prediction_dataset)
    prediction_dataset = evaluate(models, df)
    final_predictions = predict_with_params(prediction_dataset, params)

    cols = [
        's__', 'estimated_s', 'rse', 'rse_normalized',
        'rse_s0', 'rse_s1', 'rse_s2', 'rse_s3', 
        'rse_s0_normalized', 'rse_s1_normalized',
        'rse_s2_normalized', 'rse_s3_normalized'
    ] + [f'weight_{m}' for m, _ in enumerate(models)] + [f'estimated_s_model_{m}' for m, _ in enumerate(models)]

    return final_predictions[cols]

In [8]:
prediction_dataset = predicts(models, df)
prediction_dataset.head()

,s__,estimated_s,rse,rse_normalized,rse_s0,rse_s1,rse_s2,rse_s3,rse_s0_normalized,rse_s1_normalized,...,weight_0,weight_1,weight_2,weight_3,weight_4,estimated_s_model_0,estimated_s_model_1,estimated_s_model_2,estimated_s_model_3,estimated_s_model_4
469,"(-0.164, 0.929, 0.085, -0.995)","(-0.1631950578820201, 0.9279752492372257, 0.08...",0.009883,0.502596,0.000805,0.001025,0.000317,0.007736,0.546785,0.526309,...,0.690243,-0.209949,1.003486,-0.476851,0.006107,"(-0.159, 0.927, 0.082, -0.966)","(-0.154, 0.946, 0.075, -0.895)","(-0.163, 0.921, 0.082, -1.008)","(-0.167, 0.928, 0.086, -1.081)","(-0.302, 0.825, 0.418, -2.023)"
1835,"(-0.023, -0.169, 0.043, 0.603)","(-0.022795439968427367, -0.16775525716975279, ...",0.030066,0.503476,0.000205,0.001245,0.001328,0.027289,0.546151,0.526363,...,-0.059414,0.112049,0.047927,0.882020,0.012773,"(-0.024, -0.153, 0.043, 0.239)","(-0.023, -0.149, 0.044, 0.144)","(-0.023, -0.157, 0.044, 0.37)","(-0.023, -0.17, 0.044, 0.611)","(-0.02, -0.21, 0.081, 1.341)"
213,"(0.015, -0.435, 0.072, 1.299)","(0.014632360434843486, -0.41367044649888707, 0...",0.268207,0.514023,0.000368,0.021330,0.009300,0.237210,0.546323,0.531300,...,0.176765,-0.114979,0.371649,0.349248,0.194014,"(0.015, -0.414, 0.072, 0.944)","(0.018, -0.412, 0.072, 0.861)","(0.014, -0.423, 0.073, 1.059)","(0.017, -0.431, 0.073, 1.149)","(0.015, -0.413, 0.029, 1.026)"
1856,"(0.054, 0.364, -0.127, -0.959)","(0.05315232343751864, 0.3607048861291617, -0.1...",0.064593,0.504167,0.000848,0.003295,0.000834,0.059616,0.546830,0.526867,...,0.193531,0.088489,0.241578,0.474813,-0.001377,"(0.049, 0.367, -0.127, -0.92)","(0.054, 0.348, -0.118, -0.894)","(0.055, 0.364, -0.126, -0.981)","(0.054, 0.361, -0.128, -0.856)","(0.026, 0.332, -0.05, -0.873)"
324,"(-0.126, 0.001, 0.053, -0.111)","(-0.1268663638954728, 0.001977091387791829, 0....",0.040662,0.504774,0.000866,0.000977,0.002923,0.035896,0.546849,0.526297,...,0.212209,0.244034,0.302570,0.176360,0.060921,"(-0.127, 0.001, 0.052, -0.068)","(-0.125, -0.003, 0.052, 0.008)","(-0.125, 0.002, 0.054, -0.168)","(-0.128, 0.009, 0.054, -0.285)","(-0.148, 0.005, 0.104, -0.547)"


In [9]:
prediction_dataset.describe()

,rse,rse_normalized,rse_s0,rse_s1,rse_s2,rse_s3,rse_s0_normalized,rse_s1_normalized,rse_s2_normalized,rse_s3_normalized,weight_0,weight_1,weight_2,weight_3,weight_4
count,2127.000000,2127.000000,2.127000e+03,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000
mean,0.132414,0.508721,3.939583e-03,0.010685,0.004392,0.113398,0.550095,0.528684,0.511730,0.444374,0.262863,0.189261,0.237440,0.284035,0.023526
std,0.681058,0.036839,3.447568e-02,0.065648,0.031340,0.578294,0.036405,0.016138,0.075156,0.049482,1.090588,0.852949,0.981658,0.913401,0.180488
min,0.000727,0.502111,2.942471e-07,0.000002,0.000001,0.000031,0.545935,0.526058,0.501202,0.434674,-21.732677,-17.171353,-13.858801,-4.654761,-2.125770
25%,0.013143,0.502976,4.482411e-04,0.000990,0.000505,0.009315,0.546408,0.526300,0.502410,0.435468,0.039467,-0.031188,0.070065,-0.059699,-0.010452
50%,0.029889,0.503794,1.003076e-03,0.002279,0.001145,0.023987,0.546994,0.526617,0.503945,0.436723,0.308692,0.232591,0.256326,0.135349,0.000831
75%,0.081460,0.506400,2.222395e-03,0.006218,0.002652,0.067501,0.548281,0.527586,0.507558,0.440447,0.503825,0.487784,0.424497,0.443921,0.015626
max,22.874485,1.510609,1.526437e+00,2.016004,1.023576,19.213998,2.157800,1.021633,2.955818,2.078720,30.149803,6.987628,15.026376,24.871311,1.696653
